In [ ]:
import numpy as np
from bstrapping.weights.gaussian_weights import GaussianWeights

from bstrapping.bootstrap_procedures.weighted_bootstrap import WeightedBootstrap
from bstrapping.synthetic_time_series.moving_average import MovingAverage
from bstrapping.weights.auto_regressive_weights import AutoRegressiveWeights
from bstrapping.weights.moving_average import MovingAverageWeights
import pandas as pd
import matplotlib.pyplot as plt
import scienceplots

from typing import List

In [ ]:
pd.set_option('display.max_columns', None)

# Testing different bootstrap procedures for a single sample

In [ ]:
# specify variance, mean and number of the samples
mean = 4
number_sample_points = 2000

# generate samples from a normal distribution
time_series = MovingAverage(mean=mean, parameters=np.array([0.8, 0.6]))
samples = time_series.generate_samples(number_samples=number_sample_points)
print(f"Estimated mean: {np.mean(samples)}")

In [ ]:
# Benchmark
evaluation_bootstraps = []
for weights in [AutoRegressiveWeights(samples=samples),
                GaussianWeights(samples=samples),
                MovingAverageWeights(samples=samples)]:
    bootstrap = WeightedBootstrap(samples=samples, weights=weights, number_bootstrap_samples=100)
    # asymptotic variance and confidence interval
    evaluation_bootstraps.append([number_sample_points * bootstrap.bootstrapped_variance,
                                  bootstrap.bootstrapped_confidence_interval(alpha=0.05)])

evaluation_bootstraps = dict(zip(['AR', 'Multiplier', 'MA'], evaluation_bootstraps))

In [ ]:
print(
    f"True asymptotic variance: {time_series.asymptotic_variance}\nEstimated mean from samples: {np.mean(samples)}\nTrue mean: {mean}")

In [ ]:
pd.DataFrame(evaluation_bootstraps)

# Statistical evaluation of the bootstrap procedures for fixed sample size

In [ ]:
runs = 10

In [ ]:
%%capture
# Benchmark
bootstrapped_variances = [[], [], []]
for _ in range(runs):
    samples = time_series.generate_samples(number_samples=number_sample_points)
    for i, weights in enumerate([AutoRegressiveWeights(samples=samples),
                                 GaussianWeights(samples=samples),
                                 #MovingAverageWeights(samples=samples)
                                 ]):
        bootstrap = WeightedBootstrap(samples=samples, weights=weights, number_bootstrap_samples=100)
        # asymptotic variance and confidence interval
        bootstrapped_variances[i].append(number_sample_points * bootstrap.bootstrapped_variance,
                                         )

statistical_evaluation = [[np.mean(variance), np.std(variance)] for variance in bootstrapped_variances]
statistical_evaluation = dict(zip(['AR',
                                   'Multiplier',
                                   #'MA'
                                   ], statistical_evaluation))

In [ ]:
print(f"Estimated mean: {np.mean(samples)}")

In [ ]:
pd.DataFrame(statistical_evaluation, index=['mean', 'std'])

# Statistical evaluation of the bootstrap procedures for increasing sample size

In [ ]:
def evaluate_bootstraps(sample_size: int, runs: int = 100, time_series: MovingAverage = time_series,
                        alpha: float = 0.05):
    bootstrapped_variances = [[], [], []]
    mean_in_confidence_interval = [[], [], []]
    for _ in range(runs):
        samples = time_series.generate_samples(number_samples=sample_size)
        for i, weights in enumerate([AutoRegressiveWeights(samples=samples),
                                     GaussianWeights(samples=samples),
                                     #MovingAverageWeights(samples=samples)
                                     ]):
            bootstrap = WeightedBootstrap(samples=samples, weights=weights, number_bootstrap_samples=100)
            # asymptotic variance and confidence interval
            bootstrapped_variances[i].append(sample_size * bootstrap.bootstrapped_variance,
                                             )
            mean_in_confidence_interval[i].append(bootstrap.bootstrapped_confidence_interval(alpha=alpha)[0] < mean <
                                                  bootstrap.bootstrapped_confidence_interval(alpha=alpha)[1])
    return bootstrapped_variances, mean_in_confidence_interval

In [ ]:
def benchmark_bootstraps(sample_sizes: List[int], runs: int = 100):
    benchmark_var = []
    benchmark_conf = []
    for sample_size in sample_sizes:
        bootstrapped_variance, mean_in_confidence_interval = evaluate_bootstraps(sample_size=sample_size, runs=runs)
        benchmark_var.append(bootstrapped_variance)
        benchmark_conf.append(mean_in_confidence_interval)

    return benchmark_var, benchmark_conf


In [ ]:
%%capture
runs = 10
sample_sizes = [1000, 5000,]
alpha = 0.05
# of the form 0 index: sample sizes 1 index: bootstrap procedures 2 index: bootstrapped variances
plain_result, in_confidence_interval = benchmark_bootstraps(sample_sizes=sample_sizes, runs=runs)

In [ ]:
list_name_weights = ['AR',
                     'Multiplier',
                     #'MA',
                     ]
statistical_evaluation = {("mean", "Asymptotic variance"): [time_series.asymptotic_variance for
                                                            sample_size_index in range(len(plain_result))]
                          } | {
                             ("mean", name,): [np.mean(plain_result[sample_size_index][name_index]) for
                                               sample_size_index in range(len(plain_result))]

                             for name_index, name in enumerate(list_name_weights)
                         } | {
                             ("std", name,): [np.std(plain_result[sample_size_index][name_index]) for
                                              sample_size_index in range(len(plain_result))]
                             for name_index, name in enumerate(list_name_weights)
                         } | {("In confidence interval", name): [
    np.sum(in_confidence_interval[sample_size_index][name_index]) / runs for
    sample_size_index in range(len(plain_result))]
                             for name_index, name in enumerate(list_name_weights)
                         } | {
                             ("In confidence interval", "Confidence level"): [
                                 (1 - alpha) for
                                 sample_size_index in range(len(plain_result))]
                             for name_index, name in enumerate(list_name_weights)}

statistical_evaluation = pd.DataFrame(statistical_evaluation, index=sample_sizes)

In [ ]:
statistical_evaluation

In [ ]:
statistical_evaluation["mean"].plot(yerr=statistical_evaluation["std"],
                                    xlabel="Sample size",
                                    ylabel="Bootstrapped variance",
                                    ylim=[0, 8])

In [ ]:
statistical_evaluation["In confidence interval"].plot(ylabel="Coverage probability",
                                                      xlabel="Stochastic process", ylim=[0, 1])

# Statistical evaluation of the bootstrap procedures for increasing dependencies

In [ ]:
name_dependence_coefficients = [
    "iid",
    "MA(1)",
    "MA(2)",
]

dependence_coefficients = [
    np.array([0]),
    np.array([0.6]),
    np.array([0.6 ** i for i in range(1, 3)]),
]



In [ ]:
def evaluate_bootstrap_increasing_dependencies(dependence_coefficients: List[np.ndarray],
                                               sample_size: int = 100,
                                               runs: int = 100):
    true_asymptotic_variances = []
    variance = []
    benchmark_var = []
    benchmark_conf = []
    for dependence_coefficient in dependence_coefficients:
        time_series_current = MovingAverage(mean=mean, parameters=dependence_coefficient)
        true_asymptotic_variances.append(time_series_current.asymptotic_variance)
        variance.append(time_series_current.variance)
        bootstrapped_variance, mean_in_confidence_interval = evaluate_bootstraps(sample_size=sample_size, runs=runs,
                                                                                 time_series=time_series_current)
        benchmark_var.append(bootstrapped_variance)
        benchmark_conf.append(mean_in_confidence_interval)

    return benchmark_var, benchmark_conf, true_asymptotic_variances, variance

In [ ]:
%%capture
plain_result_increasing_dependencies, in_confidence_interval, true_asymptotic_variances, variance = evaluate_bootstrap_increasing_dependencies(
    sample_size=500,
    dependence_coefficients=dependence_coefficients,
    runs=runs)

In [ ]:
statistical_evaluation_dependencies = {("mean", "Variance"): variance} | {
    ("mean", "Asymptotic variance"): true_asymptotic_variances} | {
                                          ("mean", name,): [
                                              np.mean(
                                                  plain_result_increasing_dependencies[sample_size_index][name_index])
                                              for
                                              sample_size_index in range(len(plain_result_increasing_dependencies))]

                                          for name_index, name in enumerate(list_name_weights)
                                      } | {
                                          ("std", name,): [np.std(
                                              plain_result_increasing_dependencies[sample_size_index][name_index])
                                              for
                                              sample_size_index in
                                              range(len(plain_result_increasing_dependencies))]
                                          for name_index, name in enumerate(list_name_weights)
                                      } | {("In confidence interval", name): [
    np.sum(in_confidence_interval[sample_size_index][name_index]) / runs for
    sample_size_index in range(len(plain_result_increasing_dependencies))]
                                          for name_index, name in enumerate(list_name_weights)
                                      } | {
                                          ("In confidence interval", "Confidence level"): [
                                              (1 - alpha) for
                                              sample_size_index in range(len(plain_result_increasing_dependencies))]
                                          for name_index, name in enumerate(list_name_weights)}

statistical_evaluation_dependencies = pd.DataFrame(statistical_evaluation_dependencies,
                                                   index=name_dependence_coefficients)

In [ ]:
statistical_evaluation_dependencies

In [ ]:
statistical_evaluation_dependencies["mean"].plot(yerr=statistical_evaluation_dependencies["std"],
                                                 xlabel="Stochastic process", ylabel="Bootstrapped variance")

In [ ]:
statistical_evaluation_dependencies["In confidence interval"].plot(ylabel="Coverage probability",
                                                                   xlabel="Stochastic process", ylim=[0, 1])

# Statistical evaluation of the bootstrap procedures for increasing sample size and dependencies

In [ ]:
%%capture

def full_benchmark(sample_sizes: List[int], 
                   dependence_coefficients: List[np.ndarray],
                   names_dependence_coefficients: List[str], 
                   runs: int = 100,
                   alpha: float = 0.05):
    benchmark = []
    for sample_size in sample_sizes:
        for index, dependence_coefficient in enumerate(dependence_coefficients):
            time_series_current = MovingAverage(mean=mean, parameters=dependence_coefficient)
            bootstrapped_variances, mean_in_confidence_interval = evaluate_bootstraps(sample_size=sample_size,
                                                                                      alpha=alpha,
                                                                                      runs=runs,
                                                                                      time_series=time_series_current)
            bootstrapped_variances = bootstrapped_variances[:-1]  # TODO: remove this later
            mean_in_confidence_interval = mean_in_confidence_interval[:-1]  # TODO: remove this later

            benchmark.append([time_series_current.asymptotic_variance,
                              time_series_current.variance,
                              names_dependence_coefficients[index],
                              sample_size] +
                             np.mean(bootstrapped_variances, axis=1).tolist()
                             + np.std(bootstrapped_variances, axis=1).tolist()
                             + [1 - alpha]
                             + (np.sum(mean_in_confidence_interval, axis=1) / runs).tolist()
                             )

    return (pd.DataFrame(benchmark, columns=pd.MultiIndex.from_tuples([("mean", "Asymptotic variance"),
                                                                       ("Variance", ""),
                                                                       ("Stochastic process", ""),
                                                                       ("Sample size", "")] +
                                                                      [("mean", name,) for name in list_name_weights] +
                                                                      [("std", name,) for name in list_name_weights] +
                                                                      [("In confidence interval", "Confidence level")]
                                                                      +
                                                                      [("In confidence interval", name,) for name in
                                                                       list_name_weights]
                                                                      )).set_index(["Sample size"]))


In [ ]:
%%capture
benchmark = full_benchmark(sample_sizes=[500, 1000, 3000],
                           dependence_coefficients=dependence_coefficients,
                           names_dependence_coefficients=name_dependence_coefficients,
                           runs=100)

In [ ]:
benchmark

In [ ]:
plt.style.use(['science', 'ieee'])
plt.rcParams.update({'font.size': 15})
fig, a = plt.subplots(2, len(name_dependence_coefficients), figsize=(18, 12))

for index, name_dependence_coefficient in enumerate(name_dependence_coefficients):
    benchmark_wrt_dependence_coefficients = benchmark[benchmark["Stochastic process"] == name_dependence_coefficient]
    benchmark_wrt_dependence_coefficients["mean"].plot(yerr=benchmark_wrt_dependence_coefficients["std"],
                                                       xlabel="",
                                                       sharex=False,
                                                       legend=False,
                                                       #fontsize=15,
                                                       #ylabel="Bootstrapped variance",
                                                       ax=a[0][index])
    benchmark_wrt_dependence_coefficients["In confidence interval"].plot(xlabel="",
                                                                         #ylabel="Coverage probability", 
                                                                         legend=False,
                                                                         ax=a[1][index],
                                                                         #fontsize=15,
                                                                         ylim=[0, 1])

    a[0][index].set_title(name_dependence_coefficient, 
                          #fontsize=15
                          )

a[0][0].set_ylabel("Bootstrapped variance")
a[1][int(len(name_dependence_coefficients) / 2)].set_xlabel("Sample size")
a[1][0].set_ylabel("Coverage probability")
a[0][-1].legend(["Baseline"]+list_name_weights)

#plt.tight_layout()

In [ ]:
# 1000,2000,5000 0,1,2

# 10000,20000,500000 30,50,70